# chem_llm: agentic DFT band structures

`chem_llm` takes a compound and a space group and runs the whole study: it fetches the
structure from the Materials Project, reduces it to the primitive cell, builds a perturbed
structure ensemble, writes Quantum ESPRESSO inputs, runs SCF + bands + post-processing,
and plots the band structure -- writing every script itself.

Everything machine-specific (API keys, where `pw.x` lives, where models are cached, where
runs are written) comes from `config.yaml`; see `config.example.yaml`.

**A run is real DFT.** Expect roughly 10-40 minutes per compound depending on cell size,
and real API cost on the Claude backend.

## 1. Configure

One call. With no argument it searches the current directory and its parents for `chem_llm.yaml` / `config.yaml`.

In [ ]:
from chem_llm import configure

settings = configure("config.yaml")

print("config      :", settings.source)
print("backend     :", settings.model.backend, "/", settings.model.claude_model)
print("work root   :", settings.work_root)
print("doc index   :", settings.doc_index_dir)
print("QE command  :", settings.qe.shell_command("pw.x", "pw.in", "pw.out"))
print("keys loaded :", {
    "anthropic": bool(settings.anthropic_api_key),
    "materials_project": bool(settings.mp_api_key),
    "huggingface": bool(settings.hf_token),
})

## 2. Run one compound

The only required arguments are the compound and its space group. The space group symbol
is in Materials Project format: underscores for subscripts, dashes for inversion bars
(`P4_2/mnm`, `Fd-3m`).

In [ ]:
from chem_llm import run_dft_agent

result = run_dft_agent(
    "TiO2", "P4_2/mnm",
    spacegroup_number=136,
    spacegroup_name="rutile",   # optional, just reads better in the prompt
    run_name="tio2_demo",
)

print(result)

### What came back

`RunResult` reports what is actually on disk, not what the agent claimed. `ok` is true only
if the agent finished **and** all three QE stages reported `JOB DONE` **and** a band gap
could be computed from the bands output.

In [ ]:
print("ok            :", result.ok)
print("agent done    :", result.completed, f"({result.num_steps} steps, {result.runtime_seconds/60:.1f} min)")
print("SCF converged :", result.scf_converged)
print("DFT finished  :", result.dft_finished)
print("atoms         :", result.n_atoms, "| electrons:", result.n_electrons)
print("band gap      :", result.band_gap, "eV")
print("work dir      :", result.work_dir)
print("plot          :", result.plot)

In [ ]:
print(result.summary)   # the agent's own write-up: assumptions, caveats, what needs review

In [ ]:
from IPython.display import Image, display

# The agent saves a PDF; render it here if a PNG was produced too.
pngs = sorted(result.work_dir.glob("calculations/000/*.png"))
display(Image(str(pngs[0]))) if pngs else print("PDF only:", result.plot)

## 3. A sweep across space groups

Same call, different compounds -- covalent, oxide, perovskite, ionic, III-V, hexagonal.
Run this from a terminal rather than the notebook if you want them in parallel:

```bash
python sandbox/run_package_benchmark.py diamond tio2 cspbbr3 mgo
```

In [ ]:
COMPOUNDS = [
    dict(compound="C",       spacegroup="Fd-3m",    spacegroup_number=227, spacegroup_name="diamond"),
    dict(compound="Si",      spacegroup="Fd-3m",    spacegroup_number=227, spacegroup_name="diamond-structure"),
    dict(compound="MgO",     spacegroup="Fm-3m",    spacegroup_number=225, spacegroup_name="rocksalt"),
    dict(compound="GaAs",    spacegroup="F-43m",    spacegroup_number=216, spacegroup_name="zincblende"),
    dict(compound="CsPbBr3", spacegroup="Pm-3m",    spacegroup_number=221, spacegroup_name="cubic perovskite"),
    dict(compound="ZnO",     spacegroup="P6_3mc",   spacegroup_number=186, spacegroup_name="wurtzite"),
]

results = []
for spec in COMPOUNDS[:2]:            # widen the slice to run more
    results.append(run_dft_agent(**spec, run_name=spec["compound"].lower()))
    print(results[-1])

In [ ]:
import pandas as pd

pd.DataFrame([{
    "compound": r.compound, "spacegroup": r.spacegroup, "ok": r.ok,
    "gap (eV)": None if r.band_gap is None else round(r.band_gap, 3),
    "atoms": r.n_atoms, "steps": r.num_steps, "min": round(r.runtime_seconds / 60, 1),
} for r in results])

## 4. Customising

Override settings per call without touching `config.yaml`, or swap the backend to the local
Qwen model (needs a GPU and `hf_token`):

```python
settings = configure("config.yaml")
settings.model.backend = "qwen"       # local transformers model instead of the API
settings.agent.max_steps = 90         # Qwen needs more steps than Claude

run_dft_agent("TiO2", "P4_2/mnm", settings=settings)
```

The prompt itself is `chem_llm.tasks.build_task`; pass `task=...` to `run_dft_agent` to
supply your own workflow entirely.

In [ ]:
from chem_llm import build_task

print(build_task("MgO", "Fm-3m", spacegroup_number=225, spacegroup_name="rocksalt")[:700], "...")